In [3]:
import os
import urllib.request
import numpy as np
import pandas as pd
import joblib

# Set dataset path
data_path = "online_shoppers_intention.csv"

# Download if not already present
if not os.path.exists(data_path):
    url = "https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip"
    zip_path = "dataset.zip"
    urllib.request.urlretrieve(url, zip_path)
    import zipfile
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Dataset downloaded and extracted successfully.")


# Delete the zip file
file_path = "dataset.zip"

# Check if the file exists before trying to delete it
if os.path.exists(file_path):
    os.remove(file_path)

df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")
df.head()

Dataset Shape: (12330, 18)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [4]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = df.drop(columns=['Revenue'])
y = df['Revenue'].astype(int)  # 0: False, 1: True

# Stratified split to preserve class distribution (approx. 84.5% False, 15.5% True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Export the raw test partition with target for the Streamlit test file
test_df = X_test.copy()
test_df['Revenue'] = y_test
test_df.to_csv("test_data.csv", index=False)
print("test_data.csv created successfully with shape:", test_df.shape)

test_data.csv created successfully with shape: (2466, 18)


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

# Classify columns
categorical_cols = ['Month', 'VisitorType', 'Weekend']
numerical_cols = [col for col in X.columns if col not in categorical_cols]

# Build preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), categorical_cols)
    ]
)

# Import the 5 required classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

# Define model dictionary with balanced class weights where applicable
models = {
    "Logistic Regression": Pipeline([
        ('prep', preprocessor),
        ('clf', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
    ]),
    "Decision Tree": Pipeline([
        ('prep', preprocessor),
        ('clf', DecisionTreeClassifier(class_weight='balanced', max_depth=6, random_state=42))
    ]),
    "kNN": Pipeline([
        ('prep', preprocessor),
        ('clf', KNeighborsClassifier(n_neighbors=7))
    ]),
    "Naive Bayes": Pipeline([
        ('prep', preprocessor),
        # GaussianNB requires dense arrays; standard scaler preserves density
        ('clf', GaussianNB())
    ]),
    "Support Vector Machine": Pipeline([
        ('prep', preprocessor),
        # probability=True is required to compute AUC score
        ('clf', SVC(class_weight='balanced', probability=True, kernel='rbf', random_state=42))
    ])
}

In [6]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef
)

os.makedirs("model", exist_ok=True)

metrics_records = []

for name, pipeline in models.items():
    print(f"Training {name}...")
    pipeline.fit(X_train, y_train)
    
    # Save the pipeline artifact to disk
    safe_name = name.lower().replace(" ", "_")
    joblib.dump(pipeline, f"model/{safe_name}.joblib")
    
    # Predict labels and probability estimates
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    # Compute the 6 required metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)
    
    metrics_records.append({
        "ML Model Name": name,
        "Accuracy": round(acc, 4),
        "AUC": round(auc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1": round(f1, 4),
        "MCC": round(mcc, 4)
    })

# Format into a clean DataFrame
metrics_df = pd.DataFrame(metrics_records)
metrics_df

Training Logistic Regression...
Training Decision Tree...
Training kNN...
Training Naive Bayes...
Training Support Vector Machine...


/Users/mac-LTHOMA24/Projects/WILP2/ML/wilp-ml-online-shoppers-intention/.venv/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.8500,0.8963,0.5107,0.7487,0.6072,0.5330
1,Decision Tree,0.8451,0.9026,0.5000,0.8325,0.6248,0.5623
2,kNN,0.8731,0.7912,0.6561,0.3796,0.4809,0.4346
3,Naive Bayes,0.6764,0.7974,0.2977,0.8010,0.4340,0.3336
4,Support Vector Machine,0.8581,0.8861,0.5307,0.7251,0.6128,0.5381


In [7]:
import sklearn
print(sklearn.__version__)

1.9.1
